# Create Naito Foundation Awards (内藤記念科学振興財団)

Creates Naito Foundation grants from the foundation's own list of past grant
recipients ("これまでの受領者",
https://www.naito-f.or.jp/jp/joseikn/jo_index.php?data=past). The page's
year x programme selector POSTs to `db_post.php`; each fragment lists research
theme, researcher (name + position), institution + department and amount.
**3,249 grants, FY2006-2026, 100% title/PI/institution/amount** (local run
2026-09-30). Titles and programme names are in Japanese as published.

Programmes: 科学奨励金・研究助成 (KEN, 1,755), 特定研究助成金 (TOK, 546),
女性研究者研究助成金 (JYO, 313), 若手研究者海外派遣助成金 (WAK, 276, travel to present
at international meetings -> `funding_type = 'conference'`), 海外研究留学助成金
(RYU, 211, overseas research fellowships -> `'fellowship'`), 次世代育成支援研究助成金
(IKU, 120), 若手ステップアップ研究助成 (STP, 28). All are research money; nothing
is filtered out. The foundation's Naito Prize (振興賞, ~1/yr) is not included.

**Prerequisites:**
- Run `scripts/local/naito_foundation_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/naito_foundation/naito_foundation_projects.parquet`
  (§1.4 shrink guard).

**`funder_award_id` (§2.1.1):** the foundation publishes no grant number, so the key is
synthetic and stable: `NAITO-{year}-{programme}-{sha1(name|theme)[:8]}`. Citation-side
check (`openalex_awards_raw`, funder 4320309785, ~430 acknowledgement shells): they
carry other funders' ids (KAKENHI, AMED) or ad-hoc strings (`87-128`, `22205-1`); no
Naito id form exists, so no shell collapses onto these rows (0).

**Amounts:** "300万円" -> JPY 3,000,000 (`amount_jpy` in the parquet); multi-year IKU
grants show the total first ("600万円 200万円/年 3年間") and ship the total. Currency JPY.
**Dates:** only the grant year (年度) is published; `start_year` = that year and
`start_date` = Jan 1 of it (runbook §2.3 year-only pattern). No end date.

**Funder details (Path A, F4320* Crossref-registered):** funder_id `4320309785`,
metadata read from `openalex.funders.funders` (ror 02s016q17, doi 10.13039/100007428).
Not to be confused with F4320316429 "Naito Science and Engineering Foundation", a
different body.

**Priority:** `506` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.naito_foundation_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/naito_foundation/naito_foundation_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_grants FROM openalex.awards.naito_foundation_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.naito_foundation_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.naito_foundation_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320309785 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320309785;

## Step 2: Create Naito Foundation Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.naito_foundation_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320309785  -- Naito Foundation
),

g AS (
    SELECT
        r.*,
        TRY_CAST(r.year AS INT) AS yr,
        NULLIF(TRIM(r.lead_given_name), '') AS given,
        NULLIF(TRIM(r.lead_family_name), '') AS family
    FROM openalex.awards.naito_foundation_raw r
    WHERE r.funder_award_id IS NOT NULL AND TRIM(r.funder_award_id) != ''
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    TRY_CAST(g.amount_jpy AS DOUBLE) as amount,
    CASE WHEN TRY_CAST(g.amount_jpy AS DOUBLE) IS NOT NULL THEN 'JPY' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    NULLIF(TRIM(g.funding_type), '') as funding_type,
    NULLIF(TRIM(g.programme), '') as funder_scheme,
    'naito_foundation_grants' as provenance,
    CASE WHEN g.yr IS NOT NULL THEN TRY_TO_DATE(CONCAT(g.yr, '-01-01'), 'yyyy-MM-dd') END as start_date,
    CAST(NULL AS DATE) as end_date,
    g.yr as start_year,
    CAST(NULL AS INT) as end_year,
    -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
    CASE WHEN g.family IS NOT NULL THEN named_struct(
        'given_name', g.given,
        'family_name', g.family,
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', NULLIF(CONCAT_WS(' ', NULLIF(TRIM(g.institution), ''), NULLIF(TRIM(g.department), '')), ''),
            'country', CAST(NULL AS STRING),
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) END as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM g
CROSS JOIN src_funder f;

In [ ]:
%sql
-- Shape check (§2.1.1): every shipped id is the synthetic NAITO-{year}-{programme}-{hash} key.
SELECT
    COUNT(*) AS total,
    COUNT(DISTINCT id) AS distinct_ids,
    SUM(CASE WHEN funder_award_id RLIKE '^NAITO-(19|20)[0-9]{2}-[A-Z]{3}-[0-9a-f]{8}$'
              AND SUBSTR(funder_award_id, 7, 4) = CAST(start_year AS STRING) THEN 0 ELSE 1 END) AS bad_shape
FROM openalex.awards.naito_foundation_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'naito_foundation_grants' AND priority = 506;

-- Insert into openalex_awards_raw with priority.
-- Priority 506: direct-from-funder ingest of the Naito Foundation grant-recipient lists.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    506 as priority
FROM openalex.awards.naito_foundation_awards;

## Verification Queries

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, start_year, amount, currency,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.naito_foundation_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, funding_type, COUNT(*) as cnt, ROUND(AVG(amount)) as avg_jpy
FROM openalex.awards.naito_foundation_awards
GROUP BY 1, 2 ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.naito_foundation_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.family_name AS family, lead_investigator.given_name AS given, COUNT(*) AS n
FROM openalex.awards.naito_foundation_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.3 / 6.7 coverage (expect ~100% title/PI/year/amount, JPY only).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    collect_set(currency) as currencies,
    MIN(amount) as min_amount, MAX(amount) as max_amount, AVG(amount) as avg_amount,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.naito_foundation_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'naito_foundation_grants'
GROUP BY provenance, priority;